Reading and Parsing data from SQL Database:

## Creating a simple SQLite database

In [34]:
import sqlite3
import os

os.makedirs("data/databases", exist_ok = True)

In [35]:
##creating a sample database:

conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor() # cursor will point to my company.db database and will be used for insertion & deletion of records, creating tables, etc.

# Creating a table in company.db with the help of cursor:


cursor.execute('''CREATE TABLE IF NOT EXISTS EMPLOYEE (Id INTEGER PRIMARY KEY , Name TEXT, Role TEXT, Department TEXT, Salary REAL)''')

OperationalError: database is locked

In [ ]:
cursor.execute('''CREATE TABLE IF NOT EXISTS PROJECTS (Id INTEGER PRIMARY KEY, Name TEXT, Status TEXT, Budget REAL, Lead_Id INTEGER)''')

In [ ]:
##Creating data for insertion:

employees = [
    (1,"Jaywardhan Pagar", "Data Scientist", "Data Science", 150000),
    (2,"Anurag Mhaske", "Software Developer", "Development", 120000),
    (3,"Nakul Karule", "Dev-Ops Engineer", "Dev-Ops", 170000),
    (4,"Maulik Tondawal", "Data Scientist", "Data Science", 140000),
    (5,"Antriksh Soun", "Designer", "Designing", 100000)
]

projects = [
    (1,"RAG Based AI Teaching Assistant","Completed","250000",1),
    (2,"RAG Based Document Management System","Active","400000",4),
    (3,"Neural Network for Breast Cancer", "Completed","600000",1),
    (4,"Fighter Drone","Active","500000",5),
    (5,"Medical AI Assistant","Discontinued","300000",3)
]

In [ ]:
cursor.executemany('INSERT OR REPLACE INTO EMPLOYEE VALUES(?,?,?,?,?)',employees)
cursor.executemany('INSERT OR REPLACE INTO PROJECTS VALUES(?,?,?,?,?)',projects)

In [ ]:
cursor.execute("select * from EMPLOYEE")

In [ ]:
conn.commit()

conn.close()

OperationalError: database is locked

## Database content extraction

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

/home/jaywardhan/RAG_Udemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## Method1: SQL Database utility:

db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

##getting database info:

print(f"Tables: {db.get_usable_table_names()}")
print("\nTable DDL: ") # DDL: DATA DEFINITION LANGUAGE
print(db.get_table_info())

Tables: ['EMPLOYEE', 'PROJECTS']

Table DDL: 

CREATE TABLE "EMPLOYEE" (
	"Id" INTEGER, 
	"Name" TEXT, 
	"Role" TEXT, 
	"Department" TEXT, 
	"Salary" REAL, 
	PRIMARY KEY ("Id")
)

/*
3 rows from EMPLOYEE table:
Id	Name	Role	Department	Salary
1	Jaywardhan Pagar	Data Scientist	Data Science	150000.0
2	Anurag Mhaske	Software Developer	Development	120000.0
3	Nakul Karule	Dev-Ops Engineer	Dev-Ops	170000.0
*/


CREATE TABLE "PROJECTS" (
	"Id" INTEGER, 
	"Name" TEXT, 
	"Status" TEXT, 
	"Budget" REAL, 
	"Lead_Id" INTEGER, 
	PRIMARY KEY ("Id")
)

/*
3 rows from PROJECTS table:
Id	Name	Status	Budget	Lead_Id
1	RAG Based AI Teaching Assistant	Completed	250000.0	1
2	RAG Based Document Management System	Active	400000.0	4
3	Neural Network for Breast Cancer	Completed	600000.0	1
*/


In [36]:
## Custom parsing of database:

from typing import List
from langchain_core.documents import Document

def Customsql(sqlpath: str) -> List[Document]:

    """Convert SQL Database to Document Data Structures with context"""

    conn = sqlite3.connect(sqlpath)
    cursor = conn.cursor()

    documents = []

    ## Step1: Create documents for each table

    cursor.execute("SELECT name from sqlite_master WHERE type = 'table';")
    tables = cursor.fetchall()

    for table in tables:

        table_name = table[0]

        #Get table schema:

        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        column_names = [col[1] for col in columns]

        # Get table data:

        cursor.execute(f"select * from {table_name}") 
        rows = cursor.fetchall()

        #Create table overview document:

        table_content = f"Table: {table_name}"
        table_content += f"Columns: {",".join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n"

        # Add sample records:

        table_content += "Sample Records: \n"

        for row in rows[:5]:
            records = dict(zip(column_names,row))
            table_content += f"\n{records}\n"


        doc = Document(
            page_content= table_content,
            metadata = {
                'source' : sqlpath,
                'table_name' : table_name,
                'num_records' : len(rows),
                'data_type' : 'sql_table'
            }
        )

        documents.append(doc)
    conn.close()
    return documents


docs = Customsql("data/databases/company.db")
print(docs[0])



OperationalError: database is locked